In [16]:
import os
import random
import numpy as np
import torch

# =========================================================
# STEP 1: Project setup + create fresh target splits + save them
# =========================================================

# ---------------------------
# Reproducibility
# ---------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---------------------------
# Paths
# ---------------------------
BASE_OUT = r"G:\New Implemented Papers\Cutting Tool Paper"
EXP_NAME = "1320_1440"
EXP_DIR  = os.path.join(BASE_OUT, EXP_NAME)
FIG_DIR  = os.path.join(EXP_DIR, "figs")
CKPT_DIR = os.path.join(EXP_DIR, "checkpoints")
NPY_DIR  = os.path.join(EXP_DIR, "npy")
LOG_DIR  = os.path.join(EXP_DIR, "logs")
SPLIT_DIR = os.path.join(BASE_OUT, "splits")

for p in [EXP_DIR, FIG_DIR, CKPT_DIR, NPY_DIR, LOG_DIR, SPLIT_DIR]:
    os.makedirs(p, exist_ok=True)

print("EXP_DIR :", EXP_DIR)
print("FIG_DIR :", FIG_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("NPY_DIR :", NPY_DIR)
print("LOG_DIR :", LOG_DIR)
print("SPLIT_DIR:", SPLIT_DIR)

# ---------------------------
# Dataset roots
# ---------------------------
SOURCE_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1320_1"
TARGET_ROOT = r"G:\Data Sets\Cutting tool organized\Vibration_1440_1"

print("\nSOURCE_ROOT:", SOURCE_ROOT)
print("TARGET_ROOT:", TARGET_ROOT)

# ---------------------------
# Class mapping
# ---------------------------
class_names = ["BF", "GF", "N", "TF"]
class_to_idx = {c: i for i, c in enumerate(class_names)}

print("\nClasses:", class_to_idx)

# ---------------------------
# Helpers
# ---------------------------
def infer_label_from_foldername(folder_name):
    for cname in class_names:
        if folder_name.startswith(f"Vib_{cname}"):
            return cname
    raise ValueError(f"Cannot infer class from folder: {folder_name}")

def collect_source_files(source_root):
    all_files = []
    per_class = {c: 0 for c in class_names}

    for cname in class_names:
        folder = None
        for name in sorted(os.listdir(source_root)):
            if name.startswith(f"Vib_{cname}") and "1320" in name:
                folder = os.path.join(source_root, name)
                break

        if folder is None or not os.path.isdir(folder):
            raise RuntimeError(f"Could not find source folder for class {cname} in {source_root}")

        mats = sorted([
            os.path.join(folder, fn)
            for fn in os.listdir(folder)
            if fn.lower().endswith(".mat")
        ])

        per_class[cname] = len(mats)
        all_files.extend([(fp, class_to_idx[cname]) for fp in mats])

    return all_files, per_class

def collect_target_files_by_class(target_root):
    files_by_class = {c: [] for c in class_names}

    for name in sorted(os.listdir(target_root)):
        full = os.path.join(target_root, name)
        if not os.path.isdir(full):
            continue
        if not name.startswith("Vib_") or "1440" not in name:
            continue

        cname = infer_label_from_foldername(name)

        mats = sorted([
            os.path.join(full, fn)
            for fn in os.listdir(full)
            if fn.lower().endswith(".mat")
        ])
        files_by_class[cname].extend(mats)

    return files_by_class

def save_list(txt_path, items):
    with open(txt_path, "w", encoding="utf-8") as f:
        for x in items:
            f.write(str(x) + "\n")

def summarize_paths(file_list, name):
    print(f"\n{name}:")
    print(" count =", len(file_list))
    if len(file_list) > 0:
        print(" first =", file_list[0])
        print(" last  =", file_list[-1])

# ---------------------------
# Source files
# ---------------------------
source_files_with_labels, per_class_source = collect_source_files(SOURCE_ROOT)

print("\nSOURCE file counts:")
total_source = 0
for c in class_names:
    n = per_class_source.get(c, 0)
    print(f" {c}: {n}")
    total_source += n
print(" TOTAL:", total_source)

# ---------------------------
# Fresh target split creation
# Stratified, file-level, 80/20 per class
# ---------------------------
files_by_class = collect_target_files_by_class(TARGET_ROOT)

target_train_files = []
target_test_files = []

target_train_counts = {c: 0 for c in class_names}
target_test_counts = {c: 0 for c in class_names}

rng = random.Random(SEED)

print("\nCreating fresh target splits...")
for cname in class_names:
    files = sorted(files_by_class[cname])
    if len(files) == 0:
        raise RuntimeError(f"No target files found for class {cname}")

    files_copy = files[:]
    rng.shuffle(files_copy)

    n_total = len(files_copy)
    n_train = int(round(0.80 * n_total))
    n_train = max(1, min(n_train, n_total - 1))  # ensure both train and test exist
    n_test = n_total - n_train

    train_part = sorted(files_copy[:n_train])
    test_part  = sorted(files_copy[n_train:])

    target_train_files.extend(train_part)
    target_test_files.extend(test_part)

    target_train_counts[cname] = len(train_part)
    target_test_counts[cname] = len(test_part)

    print(f" {cname}: total={n_total}, train={len(train_part)}, test={len(test_part)}")

# final global sort for stable order
target_train_files = sorted(target_train_files)
target_test_files = sorted(target_test_files)

# ---------------------------
# Leakage check
# ---------------------------
train_set = set(target_train_files)
test_set = set(target_test_files)
overlap = train_set.intersection(test_set)

print("\nLeakage check:")
print(" Train unique:", len(train_set))
print(" Test unique :", len(test_set))
print(" Overlap     :", len(overlap))

if len(overlap) > 0:
    print("\nERROR: Overlapping files found.")
    for i, fp in enumerate(sorted(list(overlap))[:10]):
        print(f" overlap[{i}] =", fp)
    raise RuntimeError("Target train/test leakage detected.")
else:
    print(" No direct file overlap detected.")

# ---------------------------
# Save split files
# ---------------------------
target_train_txt = os.path.join(SPLIT_DIR, "target_train_files.txt")
target_test_txt  = os.path.join(SPLIT_DIR, "target_test_files.txt")

save_list(target_train_txt, target_train_files)
save_list(target_test_txt, target_test_files)

print("\nSaved fresh split files:")
print(" ", target_train_txt)
print(" ", target_test_txt)

# ---------------------------
# Summary
# ---------------------------
print("\nTarget train class counts:")
for c in class_names:
    print(f" {c}: {target_train_counts[c]}")

print("\nTarget test class counts:")
for c in class_names:
    print(f" {c}: {target_test_counts[c]}")

if len(source_files_with_labels) > 0:
    print("\nSOURCE example:", source_files_with_labels[0][0])

summarize_paths(target_train_files, "TARGET TRAIN")
summarize_paths(target_test_files, "TARGET TEST")

# ---------------------------
# Save config
# ---------------------------
cfg = {
    "seed": SEED,
    "source_root": SOURCE_ROOT,
    "target_root": TARGET_ROOT,
    "base_out": BASE_OUT,
    "exp_name": EXP_NAME,
    "exp_dir": EXP_DIR,
    "fig_dir": FIG_DIR,
    "ckpt_dir": CKPT_DIR,
    "npy_dir": NPY_DIR,
    "log_dir": LOG_DIR,
    "split_dir": SPLIT_DIR,
    "target_train_list": target_train_txt,
    "target_test_list": target_test_txt,
    "n_source_files": int(len(source_files_with_labels)),
    "n_target_train_files": int(len(target_train_files)),
    "n_target_test_files": int(len(target_test_files)),
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "target_train_counts": target_train_counts,
    "target_test_counts": target_test_counts
}

cfg_path = os.path.join(EXP_DIR, "run_config.npy")
np.save(cfg_path, cfg, allow_pickle=True)

print("\nSaved config to:", cfg_path)
print("\nSTEP 1 complete.")

EXP_DIR : G:\New Implemented Papers\Cutting Tool Paper\1320_1440
FIG_DIR : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs
CKPT_DIR: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\checkpoints
NPY_DIR : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\npy
LOG_DIR : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\logs
SPLIT_DIR: G:\New Implemented Papers\Cutting Tool Paper\splits

SOURCE_ROOT: G:\Data Sets\Cutting tool organized\Vibration_1320_1
TARGET_ROOT: G:\Data Sets\Cutting tool organized\Vibration_1440_1

Classes: {'BF': 0, 'GF': 1, 'N': 2, 'TF': 3}

SOURCE file counts:
 BF: 112
 GF: 111
 N: 116
 TF: 112
 TOTAL: 451

Creating fresh target splits...
 BF: total=112, train=90, test=22
 GF: total=110, train=88, test=22
 N: total=112, train=90, test=22
 TF: total=112, train=90, test=22

Leakage check:
 Train unique: 358
 Test unique : 88
 Overlap     : 0
 No direct file overlap detected.

Saved fresh split files:
  G:\New Implemented Papers\Cutting Tool 

In [17]:
# =========================================================
# STEP 2 (FINAL): Correct loader for Cutting Tool dataset
# =========================================================

import os
import numpy as np
from scipy.io import loadmat
# -----------------------------
# Helper: read file list
# -----------------------------
def read_list(path):

    with open(path, "r") as f:
        lines = f.readlines()

    files = [line.strip() for line in lines if line.strip()]

    return files

cfg = np.load(os.path.join(EXP_DIR, "run_config.npy"), allow_pickle=True).item()

target_train_files = read_list(cfg["target_train_list"])
target_test_files  = read_list(cfg["target_test_list"])

print("Target train files:", len(target_train_files))
print("Target test files :", len(target_test_files))

# -----------------------------
# Parameters
# -----------------------------
WINDOW = 1024
STRIDE = 512
CHANNEL = 0   # choose channel 0 or 1

# -----------------------------
# Load signal
# -----------------------------
def load_signal(mat_path):

    data = loadmat(mat_path)

    if "signals" not in data:
        raise RuntimeError(f"'signals' variable not found in {mat_path}")

    signals = data["signals"]      # shape (2,25600)

    x = signals[CHANNEL, :]        # select channel

    x = x.astype(np.float32)

    # normalization
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)

    return x

# -----------------------------
# Segmentation
# -----------------------------
def segment_signal(signal):

    segments = []
    N = len(signal)

    for start in range(0, N - WINDOW + 1, STRIDE):
        seg = signal[start:start+WINDOW]
        segments.append(seg)

    return np.stack(segments)

# -----------------------------
# Test loader
# -----------------------------
print("\nTesting loader")

test_file = target_train_files[0]

print("Test file:", test_file)

signal = load_signal(test_file)

segments = segment_signal(signal)

print("Signal length :", len(signal))
print("Segments shape:", segments.shape)
print("Segment mean  :", segments[0].mean())
print("Segment std   :", segments[0].std())

print("\nSTEP 2 complete.")


Target train files: 358
Target test files : 88

Testing loader
Test file: G:\Data Sets\Cutting tool organized\Vibration_1440_1\Vib_BF1440_1\20250212_133259_Vib.mat
Signal length : 25600
Segments shape: (49, 1024)
Segment mean  : -0.021440286
Segment std   : 0.87277716

STEP 2 complete.


In [18]:
# =========================================================
# STEP 3: Dataset construction + dataloaders
# =========================================================

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# -----------------------------
# Augmentation
# -----------------------------
def augment_signal(x):

    x_aug = x.copy()

    # small gaussian noise
    noise = np.random.normal(0, 0.01, size=x_aug.shape)
    x_aug += noise

    # random scaling
    scale = np.random.uniform(0.9, 1.1)
    x_aug *= scale

    return x_aug


# -----------------------------
# Label inference
# -----------------------------
def infer_label(filepath):

    folder = os.path.basename(os.path.dirname(filepath))

    if "Vib_BF" in folder:
        return 0
    if "Vib_GF" in folder:
        return 1
    if "Vib_N" in folder:
        return 2
    if "Vib_TF" in folder:
        return 3

    raise RuntimeError("Cannot infer label")


# -----------------------------
# Dataset
# -----------------------------
class CuttingToolDataset(Dataset):

    def __init__(self, file_list, use_labels=True, augment=False):

        self.samples = []
        self.labels = []
        self.augment = augment
        self.use_labels = use_labels

        for fp in file_list:

            signal = load_signal(fp)
            segments = segment_signal(signal)

            if use_labels:
                y = infer_label(fp)

            for seg in segments:

                self.samples.append(seg)

                if use_labels:
                    self.labels.append(y)

        self.samples = np.stack(self.samples).astype(np.float32)

        if use_labels:
            self.labels = np.array(self.labels).astype(np.int64)

        print("Dataset built")
        print("Total segments:", len(self.samples))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):

        x = self.samples[idx]

        if self.augment:
            x_aug = augment_signal(x)
        else:
            x_aug = x

        x = torch.tensor(x).float().unsqueeze(0)
        x_aug = torch.tensor(x_aug).float().unsqueeze(0)

        if self.use_labels:
            y = torch.tensor(self.labels[idx])
            return x, x_aug, y
        else:
            return x, x_aug


# -----------------------------
# Build datasets
# -----------------------------
print("\nBuilding datasets...")

src_dataset = CuttingToolDataset(
    [fp for fp,_ in source_files_with_labels],
    use_labels=True,
    augment=True
)

tgt_train_dataset = CuttingToolDataset(
    target_train_files,
    use_labels=False,
    augment=True
)

tgt_test_dataset = CuttingToolDataset(
    target_test_files,
    use_labels=True,
    augment=False
)

# -----------------------------
# Dataloaders
# -----------------------------
BATCH_SIZE = 128

src_loader = DataLoader(
    src_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

tgt_tr_loader = DataLoader(
    tgt_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

tgt_te_loader = DataLoader(
    tgt_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nDataloaders ready")

print("Source batches:", len(src_loader))
print("Target train batches:", len(tgt_tr_loader))
print("Target test batches:", len(tgt_te_loader))

print("\nSTEP 3 complete.")


Building datasets...
Dataset built
Total segments: 22099
Dataset built
Total segments: 17542
Dataset built
Total segments: 4312

Dataloaders ready
Source batches: 172
Target train batches: 137
Target test batches: 34

STEP 3 complete.


In [19]:
# =========================================================
# STEP 4: Model definition + sanity forward pass
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Feature extractor
# -----------------------------
class FeatureExtractor1D(nn.Module):
    def __init__(self, feat_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=1, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Linear(256, feat_dim)

    def forward(self, x):
        x = self.features(x)           # (B,256,1)
        x = x.squeeze(-1)              # (B,256)
        feat = self.fc(x)              # (B,feat_dim)
        feat = F.normalize(feat, p=2, dim=1)
        return feat

# -----------------------------
# Classifier
# -----------------------------
class ClassifierHead(nn.Module):
    def __init__(self, feat_dim=128, n_classes=4):
        super().__init__()
        self.fc = nn.Linear(feat_dim, n_classes)

    def forward(self, feat):
        return self.fc(feat)

# -----------------------------
# Full model
# -----------------------------
class FaultNet(nn.Module):
    def __init__(self, feat_dim=128, n_classes=4):
        super().__init__()
        self.backbone = FeatureExtractor1D(feat_dim=feat_dim)
        self.classifier = ClassifierHead(feat_dim=feat_dim, n_classes=n_classes)

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.classifier(feat)
        return feat, logits

# -----------------------------
# Instantiate model
# -----------------------------
model = FaultNet(feat_dim=128, n_classes=4).to(device)

# -----------------------------
# Sanity forward pass
# -----------------------------
xb, xb_aug, yb = next(iter(src_loader))
xb = xb.to(device)
xb_aug = xb_aug.to(device)

with torch.no_grad():
    feat, logits = model(xb)
    feat_aug, logits_aug = model(xb_aug)

print("\nSanity forward pass:")
print("Input shape       :", xb.shape)
print("Feature shape     :", feat.shape)
print("Logits shape      :", logits.shape)
print("Aug feature shape :", feat_aug.shape)
print("Aug logits shape  :", logits_aug.shape)

print("\nSTEP 4 complete.")

Device: cpu

Sanity forward pass:
Input shape       : torch.Size([128, 1, 1024])
Feature shape     : torch.Size([128, 128])
Logits shape      : torch.Size([128, 4])
Aug feature shape : torch.Size([128, 128])
Aug logits shape  : torch.Size([128, 4])

STEP 4 complete.


In [20]:
# =========================================================
# STEP 5: Source pretraining
# =========================================================

import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Optimizer
# -----------------------------
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS_SRC = 12

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# -----------------------------
# Evaluation function
# -----------------------------
@torch.no_grad()
def evaluate_classifier(model, loader, has_labels=True):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0

    for batch in loader:
        if has_labels:
            x, x_aug, y = batch
            x = x.to(device)
            y = y.to(device)

            feat, logits = model(x)
            loss = F.cross_entropy(logits, y)

            pred = torch.argmax(logits, dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)
            total_loss += loss.item() * y.size(0)

    acc = correct / max(total, 1)
    avg_loss = total_loss / max(total, 1)

    return avg_loss, acc

# -----------------------------
# Training loop
# -----------------------------
best_target_acc = 0.0
best_ckpt_path = os.path.join(CKPT_DIR, "best_source_pretrain.pt")

print("\n==============================")
print("SOURCE PRETRAINING START")
print("==============================")

for epoch in range(1, EPOCHS_SRC + 1):

    model.train()
    running_loss = 0.0
    total_seen = 0

    pbar = tqdm(src_loader, desc=f"Epoch {epoch:02d}/{EPOCHS_SRC}", leave=False)

    for x, x_aug, y in pbar:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        feat, logits = model(x)
        loss = F.cross_entropy(logits, y)

        loss.backward()
        optimizer.step()

        bs = y.size(0)
        running_loss += loss.item() * bs
        total_seen += bs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / max(total_seen, 1)

    # evaluate directly on target test as baseline
    target_loss, target_acc = evaluate_classifier(model, tgt_te_loader, has_labels=True)

    print(f"Epoch [{epoch:02d}/{EPOCHS_SRC}]  "
          f"Train CE={train_loss:.4f}  |  "
          f"TargetTest CE={target_loss:.4f}  |  "
          f"TargetTest Acc={target_acc*100:.2f}%")

    if target_acc > best_target_acc:
        best_target_acc = target_acc
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "best_target_acc": best_target_acc
        }, best_ckpt_path)

print("\nBest target-test accuracy during source pretraining:", f"{best_target_acc*100:.2f}%")
print("Saved best checkpoint to:", best_ckpt_path)

print("\nSTEP 5 complete.")

Device: cpu

SOURCE PRETRAINING START


Epoch [01/12]  Train CE=0.5359  |  TargetTest CE=0.5496  |  TargetTest Acc=79.55%


Epoch [02/12]  Train CE=0.1313  |  TargetTest CE=0.1314  |  TargetTest Acc=98.47%


Epoch [03/12]  Train CE=0.0543  |  TargetTest CE=0.0599  |  TargetTest Acc=99.44%


Epoch [04/12]  Train CE=0.0307  |  TargetTest CE=0.0515  |  TargetTest Acc=99.17%


...

Epoch [08/12]  Train CE=0.0090  |  TargetTest CE=0.0237  |  TargetTest Acc=99.56%


Epoch [09/12]  Train CE=0.0072  |  TargetTest CE=0.0278  |  TargetTest Acc=99.33%


Epoch [10/12]  Train CE=0.0061  |  TargetTest CE=0.0316  |  TargetTest Acc=99.07%


Epoch [11/12]  Train CE=0.0051  |  TargetTest CE=0.0158  |  TargetTest Acc=99.68%


Epoch [12/12]  Train CE=0.0044  |  TargetTest CE=0.0175  |  TargetTest Acc=99.58%

Best target-test accuracy during source pretraining: 99.68%
Saved best checkpoint to: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\checkpoints\best_source_pretrain.pt

STEP 5 complete.


In [21]:
# =========================================================
# STEP 6 FIX: Re-extract target features for ALL target-train samples
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model.eval()

# make a loader that does NOT drop the last batch
tgt_tr_loader_full = DataLoader(
    tgt_train_dataset,
    batch_size=128,
    shuffle=False,
    drop_last=False
)

all_feats = []
all_probs = []

print("\nExtracting target train features for ALL samples...")

with torch.no_grad():
    for xb, xb_aug in tgt_tr_loader_full:
        xb = xb.to(device)

        feat, logits = model(xb)
        prob = F.softmax(logits, dim=1)

        all_feats.append(feat.cpu().numpy())
        all_probs.append(prob.cpu().numpy())

features = np.concatenate(all_feats, axis=0)
probs = np.concatenate(all_probs, axis=0)

print("\nFeature matrix shape     :", features.shape)
print("Probability matrix shape :", probs.shape)
print("Expected sample count    :", len(tgt_train_dataset.samples))

np.save(os.path.join(NPY_DIR, "target_features.npy"), features)
np.save(os.path.join(NPY_DIR, "target_probs.npy"), probs)

print("\nSaved:")
print("target_features.npy")
print("target_probs.npy")

print("\nSTEP 6 FIX complete.")

Device: cpu

Extracting target train features for ALL samples...

Feature matrix shape     : (17542, 128)
Probability matrix shape : (17542, 4)
Expected sample count    : 17542

Saved:
target_features.npy
target_probs.npy

STEP 6 FIX complete.


In [22]:
# =========================================================
# STEP 7: SSCKNN pseudo label generation
# =========================================================

import numpy as np
from sklearn.neighbors import NearestNeighbors

print("\nGenerating pseudo labels using SSCKNN...")

features = np.load(os.path.join(NPY_DIR, "target_features.npy"))
probs = np.load(os.path.join(NPY_DIR, "target_probs.npy"))

print("Features shape:", features.shape)
print("Probabilities shape:", probs.shape)

# -----------------------------
# KNN parameters
# -----------------------------
K = 5

nbrs = NearestNeighbors(n_neighbors=K+1, metric="cosine")
nbrs.fit(features)

distances, indices = nbrs.kneighbors(features)

pseudo_labels = np.zeros(len(features), dtype=np.int64)
pseudo_conf = np.zeros(len(features), dtype=np.float32)

for i in range(len(features)):

    neigh_idx = indices[i][1:]      # exclude itself

    neigh_probs = probs[neigh_idx]

    mean_prob = neigh_probs.mean(axis=0)

    pseudo_labels[i] = np.argmax(mean_prob)

    pseudo_conf[i] = np.max(mean_prob)

# -----------------------------
# Save pseudo labels
# -----------------------------
np.save(os.path.join(NPY_DIR, "pseudo_labels.npy"), pseudo_labels)
np.save(os.path.join(NPY_DIR, "pseudo_confidence.npy"), pseudo_conf)

print("\nPseudo labels generated.")

print("\nPseudo label distribution:")

for c in range(4):
    count = (pseudo_labels == c).sum()
    print(f"Class {c}: {count}")

print("\nConfidence statistics")
print("Mean confidence:", pseudo_conf.mean())
print("Min confidence :", pseudo_conf.min())
print("Max confidence :", pseudo_conf.max())

print("\nSaved:")
print("pseudo_labels.npy")
print("pseudo_confidence.npy")

print("\nSTEP 7 complete.")


Generating pseudo labels using SSCKNN...
Features shape: (17542, 128)
Probabilities shape: (17542, 4)

Pseudo labels generated.

Pseudo label distribution:
Class 0: 4351
Class 1: 4371
Class 2: 4410
Class 3: 4410

Confidence statistics
Mean confidence: 0.9900952
Min confidence : 0.41166076
Max confidence : 0.9963622

Saved:
pseudo_labels.npy
pseudo_confidence.npy

STEP 7 complete.


In [23]:
# =========================================================
# STEP 8 (REVISED): Confidence-weighted target adaptation
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Load pseudo labels and confidence
# -----------------------------
pseudo_labels = np.load(os.path.join(NPY_DIR, "pseudo_labels.npy")).astype(np.int64)
pseudo_conf   = np.load(os.path.join(NPY_DIR, "pseudo_confidence.npy")).astype(np.float32)

print("Pseudo labels shape :", pseudo_labels.shape)
print("Pseudo conf shape   :", pseudo_conf.shape)
print("Mean confidence     :", float(pseudo_conf.mean()))
print("Min confidence      :", float(pseudo_conf.min()))
print("Max confidence      :", float(pseudo_conf.max()))

# -----------------------------
# Build weighted pseudo dataset
# -----------------------------
class WeightedPseudoDataset(Dataset):
    def __init__(self, samples, labels, weights):
        self.samples = samples.astype(np.float32)
        self.labels = labels.astype(np.int64)
        self.weights = weights.astype(np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = torch.tensor(self.samples[idx]).float().unsqueeze(0)
        y = torch.tensor(self.labels[idx]).long()
        w = torch.tensor(self.weights[idx]).float()
        return x, y, w

# use all target-train segments
tgt_samples = tgt_train_dataset.samples
assert len(tgt_samples) == len(pseudo_labels) == len(pseudo_conf), \
    f"Mismatch: samples={len(tgt_samples)}, labels={len(pseudo_labels)}, conf={len(pseudo_conf)}"

pseudo_dataset = WeightedPseudoDataset(
    samples=tgt_samples,
    labels=pseudo_labels,
    weights=pseudo_conf
)

pseudo_loader = DataLoader(
    pseudo_dataset,
    batch_size=128,
    shuffle=True,
    drop_last=True
)

print("\nWeighted pseudo dataset size:", len(pseudo_dataset))
print("Pseudo loader batches       :", len(pseudo_loader))

# -----------------------------
# Load best source-pretrained model
# -----------------------------
src_ckpt_path = os.path.join(CKPT_DIR, "best_source_pretrain.pt")
ckpt = torch.load(src_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)

# -----------------------------
# Optimizer
# -----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
EPOCHS_ADAPT = 6

best_acc = 0.0
best_path = os.path.join(CKPT_DIR, "weighted_stage2_best.pt")

# -----------------------------
# Evaluation function
# -----------------------------
@torch.no_grad()
def evaluate_target(model, loader):
    model.eval()
    correct = 0
    total = 0

    for xb, xb_aug, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model(xb)
        pred = torch.argmax(logits, dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    return correct / max(total, 1)

# -----------------------------
# Training
# -----------------------------
print("\n==============================")
print("CONFIDENCE-WEIGHTED ADAPTATION START")
print("==============================")

for epoch in range(1, EPOCHS_ADAPT + 1):
    model.train()

    total_loss = 0.0
    total_seen = 0

    pbar = tqdm(pseudo_loader, desc=f"Epoch {epoch:02d}/{EPOCHS_ADAPT}", leave=False)

    for x, y, w in pbar:
        x = x.to(device)
        y = y.to(device)
        w = w.to(device)

        optimizer.zero_grad()

        feat, logits = model(x)

        # per-sample CE
        ce_each = F.cross_entropy(logits, y, reduction="none")

        # confidence-weighted CE
        loss = (w * ce_each).mean()

        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_seen += bs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / max(total_seen, 1)
    acc = evaluate_target(model, tgt_te_loader)

    print(f"Epoch [{epoch:02d}/{EPOCHS_ADAPT}]  Weighted CE={train_loss:.4f}  |  Target Acc={acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(
            {
                "model_state": model.state_dict(),
                "best_acc": best_acc,
                "epoch": epoch
            },
            best_path
        )

print("\nBest weighted adaptation accuracy:", f"{best_acc*100:.2f}%")
print("Saved best model to:", best_path)

print("\nSTEP 8 complete.")

C:\Users\saifu\AppData\Local\Temp\ipykernel_24196\796173002.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(src_ckpt_path, map_location=device)


Device: cpu
Pseudo labels shape : (17542,)
Pseudo conf shape   : (17542,)
Mean confidence     : 0.9900951981544495
Min confidence      : 0.411660760641098
Max confidence      : 0.9963622093200684

Weighted pseudo dataset size: 17542
Pseudo loader batches       : 137

CONFIDENCE-WEIGHTED ADAPTATION START


Epoch [01/6]  Weighted CE=0.0112  |  Target Acc=98.42%


Epoch [02/6]  Weighted CE=0.0078  |  Target Acc=99.95%


Epoch [03/6]  Weighted CE=0.0094  |  Target Acc=99.77%


Epoch [04/6]  Weighted CE=0.0078  |  Target Acc=99.98%


Epoch [05/6]  Weighted CE=0.0078  |  Target Acc=99.84%


Epoch [06/6]  Weighted CE=0.0072  |  Target Acc=97.19%

Best weighted adaptation accuracy: 99.98%
Saved best model to: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\checkpoints\weighted_stage2_best.pt

STEP 8 complete.


In [24]:
# =========================================================
# STEP 9: Final evaluation + HQ figures + t-SNE metrics
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Paths
# -----------------------------
BEST_MODEL_PATH = os.path.join(CKPT_DIR, "weighted_stage2_best.pt")
OUT_EVAL_DIR = os.path.join(FIG_DIR, "final_eval")
os.makedirs(OUT_EVAL_DIR, exist_ok=True)

print("Best model path:", BEST_MODEL_PATH)
print("Eval output dir:", OUT_EVAL_DIR)

# -----------------------------
# Load best adapted model
# -----------------------------
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)
model.eval()

# -----------------------------
# Collect outputs on target test
# -----------------------------
all_ytrue = []
all_ypred = []
all_yprob = []
all_feat = []

with torch.no_grad():
    for xb, xb_aug, yb in tgt_te_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model(xb)
        prob = F.softmax(logits, dim=1)
        pred = torch.argmax(prob, dim=1)

        all_ytrue.append(yb.cpu().numpy())
        all_ypred.append(pred.cpu().numpy())
        all_yprob.append(prob.cpu().numpy())
        all_feat.append(feat.cpu().numpy())

y_true = np.concatenate(all_ytrue, axis=0).astype(np.int64)
y_pred = np.concatenate(all_ypred, axis=0).astype(np.int64)
y_prob = np.concatenate(all_yprob, axis=0).astype(np.float32)
feat_te = np.concatenate(all_feat, axis=0).astype(np.float32)

print("\nCollected outputs:")
print("y_true shape :", y_true.shape)
print("y_pred shape :", y_pred.shape)
print("y_prob shape :", y_prob.shape)
print("feat shape   :", feat_te.shape)

acc = (y_true == y_pred).mean()
print(f"Target test accuracy: {acc*100:.4f}%")

# save raw arrays
np.save(os.path.join(OUT_EVAL_DIR, "y_true.npy"), y_true)
np.save(os.path.join(OUT_EVAL_DIR, "y_pred.npy"), y_pred)
np.save(os.path.join(OUT_EVAL_DIR, "y_prob.npy"), y_prob)
np.save(os.path.join(OUT_EVAL_DIR, "feat_test.npy"), feat_te)

# -----------------------------
# Class names
# -----------------------------
class_names = ["BF", "GF", "N", "TF"]
n_classes = len(class_names)

# -----------------------------
# Classification report
# -----------------------------
rep = classification_report(y_true, y_pred, target_names=class_names, digits=6)
rep_path = os.path.join(OUT_EVAL_DIR, "classification_report.txt")
with open(rep_path, "w", encoding="utf-8") as f:
    f.write(rep)

print("\nClassification report saved to:")
print(rep_path)

# -----------------------------
# Confusion Matrix (HQ)
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

cm_fig_path = os.path.join(OUT_EVAL_DIR, "confusion_matrix_1200dpi.png")
plt.figure(figsize=(6, 6))
plt.imshow(cm, cmap="YlGnBu")

ticks = np.arange(n_classes)
plt.xticks(ticks, class_names, fontsize=16, fontweight="bold")
plt.yticks(ticks, class_names, fontsize=16, fontweight="bold")

plt.gca().set_xticks(np.arange(-.5, n_classes, 1), minor=True)
plt.gca().set_yticks(np.arange(-.5, n_classes, 1), minor=True)
plt.grid(which="minor", color="black", linestyle="-", linewidth=1.5)
plt.tick_params(which="minor", bottom=False, left=False)

thresh = cm.max() * 0.5 if cm.max() > 0 else 0.5
for i in range(n_classes):
    for j in range(n_classes):
        plt.text(
            j, i, f"{cm[i, j]}",
            ha="center", va="center",
            fontsize=18, fontweight="bold",
            color="white" if cm[i, j] > thresh else "black"
        )

plt.xlabel("Predicted Class", fontsize=18, fontweight="bold")
plt.ylabel("True Class", fontsize=18, fontweight="bold")
plt.tight_layout()
plt.savefig(cm_fig_path, dpi=1200, bbox_inches="tight")
plt.close()

print("\nConfusion matrix saved to:")
print(cm_fig_path)

# -----------------------------
# -----------------------------
# ROC Curves (HQ with different line styles)
# -----------------------------
y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))

roc_fig_path = os.path.join(OUT_EVAL_DIR, "roc_curves_1200dpi.png")
auc_txt_path = os.path.join(OUT_EVAL_DIR, "auc_scores.txt")

plt.figure(figsize=(6, 6))

# different line styles
line_styles = ["-", "--", ":", "-."]
colors = ["blue", "red", "green", "purple"]

auc_values = []

for c in range(n_classes):

    fpr, tpr, _ = roc_curve(y_true_bin[:, c], y_prob[:, c])
    roc_auc = auc(fpr, tpr)
    auc_values.append(roc_auc)

    plt.plot(
        fpr,
        tpr,
        linestyle=line_styles[c],
        color=colors[c],
        linewidth=3,
        label=f"{class_names[c]} (AUC = {roc_auc:.4f})"
    )

# chance line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    linewidth=2,
    label="Chance"
)

plt.xlabel("False Positive Rate", fontsize=16, fontweight="bold")
plt.ylabel("True Positive Rate", fontsize=16, fontweight="bold")

leg = plt.legend(fontsize=12, frameon=True)
leg.get_frame().set_edgecolor("black")
leg.get_frame().set_linewidth(1.5)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(roc_fig_path, dpi=1200, bbox_inches="tight")
plt.close()

with open(auc_txt_path, "w") as f:
    for i, cname in enumerate(class_names):
        f.write(f"{cname}: {auc_values[i]:.6f}\n")

print("ROC curves saved:", roc_fig_path)

# -----------------------------
# t-SNE (HQ)
# -----------------------------
print("\nRunning t-SNE...")

tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)

Z = tsne.fit_transform(feat_te)

tsne_fig_path = os.path.join(OUT_EVAL_DIR, "tsne_1200dpi.png")
markers = ["o", "s", "^", "D"]

plt.figure(figsize=(6, 6))
for c, cname in enumerate(class_names):
    idx = (y_true == c)
    plt.scatter(
        Z[idx, 0], Z[idx, 1],
        s=70,
        marker=markers[c],
        edgecolors="black",
        linewidths=0.8,
        alpha=0.9,
        label=cname
    )

plt.xlabel("t-SNE Dimension 1", fontsize=16, fontweight="bold")
plt.ylabel("t-SNE Dimension 2", fontsize=16, fontweight="bold")
leg = plt.legend(fontsize=12, frameon=True)
leg.get_frame().set_edgecolor("black")
leg.get_frame().set_linewidth(1.5)
plt.tight_layout()
plt.savefig(tsne_fig_path, dpi=1200, bbox_inches="tight")
plt.close()

print("t-SNE figure saved to:")
print(tsne_fig_path)

# save embedding
np.save(os.path.join(OUT_EVAL_DIR, "tsne_embedding.npy"), Z)

# -----------------------------
# t-SNE quality metrics
# -----------------------------
# on 2D t-SNE
sil_tsne = silhouette_score(Z, y_true)
dbi_tsne = davies_bouldin_score(Z, y_true)
chi_tsne = calinski_harabasz_score(Z, y_true)

# on original 128D features
sil_feat = silhouette_score(feat_te, y_true)
dbi_feat = davies_bouldin_score(feat_te, y_true)
chi_feat = calinski_harabasz_score(feat_te, y_true)

tsne_metrics_path = os.path.join(OUT_EVAL_DIR, "tsne_and_feature_metrics.txt")
with open(tsne_metrics_path, "w", encoding="utf-8") as f:
    f.write("=== Metrics on 2D t-SNE Embedding ===\n")
    f.write(f"Silhouette Score: {sil_tsne:.6f}\n")
    f.write(f"Davies-Bouldin Index: {dbi_tsne:.6f}\n")
    f.write(f"Calinski-Harabasz Score: {chi_tsne:.6f}\n\n")

    f.write("=== Metrics on Original 128D Features ===\n")
    f.write(f"Silhouette Score: {sil_feat:.6f}\n")
    f.write(f"Davies-Bouldin Index: {dbi_feat:.6f}\n")
    f.write(f"Calinski-Harabasz Score: {chi_feat:.6f}\n")

print("\nt-SNE / feature metrics:")
print(f"2D t-SNE  -> Silhouette={sil_tsne:.6f}, DBI={dbi_tsne:.6f}, CHI={chi_tsne:.6f}")
print(f"128D feat -> Silhouette={sil_feat:.6f}, DBI={dbi_feat:.6f}, CHI={chi_feat:.6f}")

print("\nMetrics saved to:")
print(tsne_metrics_path)

print("\nSTEP 9 complete.")

Device: cpu
Best model path: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\checkpoints\weighted_stage2_best.pt
Eval output dir: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval


C:\Users\saifu\AppData\Local\Temp\ipykernel_24196\766666420.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(BEST_MODEL_PATH, map_location=device)



Collected outputs:
y_true shape : (4312,)
y_pred shape : (4312,)
y_prob shape : (4312, 4)
feat shape   : (4312, 128)
Target test accuracy: 99.9768%

Classification report saved to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval\classification_report.txt

Confusion matrix saved to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval\confusion_matrix_1200dpi.png
ROC curves saved: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval\roc_curves_1200dpi.png

Running t-SNE...
t-SNE figure saved to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval\tsne_1200dpi.png

t-SNE / feature metrics:
2D t-SNE  -> Silhouette=0.493688, DBI=0.830357, CHI=5239.178176
128D feat -> Silhouette=0.873266, DBI=0.178480, CHI=50551.176261

Metrics saved to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\final_eval\tsne_and_feature_metrics.txt

STEP 9 complete.


In [25]:
# =========================================================
# STEP 10 (UPGRADED): SNR robustness experiment from 20 dB to 40 dB
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Load best adapted model
# -----------------------------
BEST_MODEL_PATH = os.path.join(CKPT_DIR, "weighted_stage2_best.pt")
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)
model.eval()

# -----------------------------
# Output folder
# -----------------------------
SNR_DIR = os.path.join(FIG_DIR, "snr_experiments_20_to_40db")
os.makedirs(SNR_DIR, exist_ok=True)

print("SNR output dir:", SNR_DIR)

# -----------------------------
# Noise function
# -----------------------------
def add_awgn(signal, snr_db):
    signal = signal.astype(np.float32)

    if snr_db is None:
        return signal.copy()

    sig_power = np.mean(signal ** 2)
    snr_linear = 10 ** (snr_db / 10.0)
    noise_power = sig_power / (snr_linear + 1e-12)

    noise = np.random.normal(0, np.sqrt(noise_power), size=signal.shape).astype(np.float32)
    noisy = signal + noise
    return noisy

# -----------------------------
# Noisy target-test dataset
# -----------------------------
class NoisyTargetTestDataset(Dataset):
    def __init__(self, base_samples, base_labels, snr_db=None):
        self.samples = base_samples
        self.labels = base_labels
        self.snr_db = snr_db

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx].copy()
        y = self.labels[idx]

        x = add_awgn(x, self.snr_db)
        x = torch.tensor(x).float().unsqueeze(0)
        y = torch.tensor(y).long()

        return x, y

# -----------------------------
# Prepare clean target test segments
# -----------------------------
test_samples = tgt_test_dataset.samples
test_labels = tgt_test_dataset.labels

print("\nTarget test samples:", len(test_samples))
print("Target test labels :", len(test_labels))

# -----------------------------
# Evaluate for each SNR
# -----------------------------
snr_levels = [20, 25, 30, 35, 40, None]
snr_names = ["20 dB","25 dB","30 dB", "35 dB", "40 dB", "Clean"]

results = []

@torch.no_grad()
def eval_noisy_loader(loader):
    model.eval()
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model(xb)
        pred = torch.argmax(logits, dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    return correct / max(total, 1)

print("\nRunning SNR evaluations...")

for snr_db, snr_name in zip(snr_levels, snr_names):
    ds = NoisyTargetTestDataset(test_samples, test_labels, snr_db=snr_db)
    ld = DataLoader(ds, batch_size=128, shuffle=False)

    acc = eval_noisy_loader(ld)
    results.append((snr_name, acc))

    print(f"{snr_name:>6}  ->  Accuracy = {acc*100:.4f}%")

# -----------------------------
# Save text results
# -----------------------------
snr_txt = os.path.join(SNR_DIR, "snr_results_20_to_40db.txt")
with open(snr_txt, "w", encoding="utf-8") as f:
    for snr_name, acc in results:
        f.write(f"{snr_name}: {acc*100:.6f}%\n")

print("\nSaved text results to:")
print(snr_txt)

# -----------------------------
# Plot high-quality SNR figure
# Correct ordering: 20 dB -> 30 dB -> 40 dB -> Clean
# Accuracy should generally increase as SNR increases
# -----------------------------
x_labels = [x[0] for x in results]
y_vals = [x[1] * 100 for x in results]

snr_fig = os.path.join(SNR_DIR, "snr_accuracy_curve_20_to_40db_1200dpi.png")

plt.figure(figsize=(8, 6))
plt.plot(
    x_labels,
    y_vals,
    marker="o",
    linestyle="-",
    linewidth=3,
    markersize=8
)

for i, v in enumerate(y_vals):
    plt.text(i, v + 0.1, f"{v:.2f}", ha="center", fontsize=12, fontweight="bold")

plt.xlabel("Noise Condition", fontsize=16, fontweight="bold")
plt.ylabel("Accuracy (%)", fontsize=16, fontweight="bold")
plt.ylim(max(0, min(y_vals) - 3), 101)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(snr_fig, dpi=1200, bbox_inches="tight")
plt.close()

print("\nSaved SNR figure to:")
print(snr_fig)

print("\nSTEP 10 upgraded complete.")

Device: cpu
SNR output dir: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\snr_experiments_20_to_40db

Target test samples: 4312
Target test labels : 4312

Running SNR evaluations...


C:\Users\saifu\AppData\Local\Temp\ipykernel_24196\3568705659.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(BEST_MODEL_PATH, map_location=device)


 20 dB  ->  Accuracy = 86.3636%
 25 dB  ->  Accuracy = 97.9128%
 30 dB  ->  Accuracy = 99.7449%
 35 dB  ->  Accuracy = 99.9072%
 40 dB  ->  Accuracy = 99.9768%
 Clean  ->  Accuracy = 99.9768%

Saved text results to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\snr_experiments_20_to_40db\snr_results_20_to_40db.txt

Saved SNR figure to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\snr_experiments_20_to_40db\snr_accuracy_curve_20_to_40db_1200dpi.png

STEP 10 upgraded complete.


In [26]:
# =========================================================
# STEP 11 (FIXED): Small labeled source data experiment
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

def train_source_with_subset(indices):
    subset_dataset = Subset(src_dataset, indices)

    loader = DataLoader(
        subset_dataset,
        batch_size=128,
        shuffle=True,
        drop_last=True
    )

    # FIX: use the actual model class you defined
    model_local = FaultNet(feat_dim=128, n_classes=4).to(device)
    optimizer = torch.optim.Adam(model_local.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(6):
        model_local.train()

        for xb, xb_aug, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            feat, logits = model_local(xb)
            loss = F.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model_local

@torch.no_grad()
def evaluate_model(model_eval):
    model_eval.eval()

    correct = 0
    total = 0

    for xb, xb_aug, yb in tgt_te_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model_eval(xb)
        pred = torch.argmax(logits, dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    return correct / max(total, 1)

print("\nRunning small labeled data experiment...\n")

rng = np.random.RandomState(42)

for frac in fractions:
    n = int(len(src_dataset) * frac)
    indices = rng.choice(len(src_dataset), n, replace=False)

    print(f"Training with {int(frac*100)}% labeled source data ({n} samples)")

    model_tmp = train_source_with_subset(indices)
    acc = evaluate_model(model_tmp)

    print(f"Target Accuracy = {acc*100:.4f}%\n")
    results.append((frac, acc))

# -----------------------------
# Save text results
# -----------------------------
txt_path = os.path.join(FIG_DIR, "small_source_data_results.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    for frac, acc in results:
        f.write(f"{int(frac*100)}%: {acc*100:.6f}%\n")

# -----------------------------
# Plot results
# -----------------------------
x = [int(r[0] * 100) for r in results]
y = [r[1] * 100 for r in results]

plt.figure(figsize=(8, 6))
plt.plot(x, y, marker='o', linewidth=3)

for i, v in enumerate(y):
    plt.text(x[i], v + 0.2, f"{v:.2f}", ha='center', fontsize=12, fontweight='bold')

plt.xlabel("Labeled Source Data (%)", fontsize=16, fontweight='bold')
plt.ylabel("Target Accuracy (%)", fontsize=16, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()

save_path = os.path.join(FIG_DIR, "small_source_data_curve_1200dpi.png")
plt.savefig(save_path, dpi=1200, bbox_inches="tight")
plt.close()

print("Saved figure:", save_path)
print("Saved text  :", txt_path)

print("\nSTEP 11 complete.")

Device: cpu

Running small labeled data experiment...

Training with 20% labeled source data (4419 samples)
Target Accuracy = 82.7922%

Training with 40% labeled source data (8839 samples)
Target Accuracy = 99.6985%

Training with 60% labeled source data (13259 samples)
Target Accuracy = 99.6289%

Training with 80% labeled source data (17679 samples)
Target Accuracy = 100.0000%

Training with 100% labeled source data (22099 samples)
Target Accuracy = 99.7217%

Saved figure: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\small_source_data_curve_1200dpi.png
Saved text  : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\small_source_data_results.txt

STEP 11 complete.


In [27]:
# =========================================================
# STEP 12: K-Fold Cross Validation on source dataset
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Prepare labels from source dataset
# -----------------------------
X_dummy = np.arange(len(src_dataset))
y_src = src_dataset.labels

print("Total source samples:", len(X_dummy))
print("Total source labels :", len(y_src))

# -----------------------------
# KFold setup
# -----------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_results = []

@torch.no_grad()
def evaluate_loader(model_eval, loader):
    model_eval.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for xb, xb_aug, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model_eval(xb)
        loss = F.cross_entropy(logits, yb)

        pred = torch.argmax(logits, dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
        total_loss += loss.item() * yb.size(0)

    acc = correct / max(total, 1)
    avg_loss = total_loss / max(total, 1)
    return avg_loss, acc

print("\nRunning 5-Fold Cross Validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_dummy, y_src), start=1):
    print(f"Fold {fold}/{N_SPLITS}")

    train_subset = Subset(src_dataset, train_idx)
    val_subset   = Subset(src_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_subset, batch_size=128, shuffle=False)

    model_fold = FaultNet(feat_dim=128, n_classes=4).to(device)
    optimizer = torch.optim.Adam(model_fold.parameters(), lr=1e-3, weight_decay=1e-4)

    EPOCHS_FOLD = 6

    for epoch in range(EPOCHS_FOLD):
        model_fold.train()

        for xb, xb_aug, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            feat, logits = model_fold(xb)
            loss = F.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    val_loss, val_acc = evaluate_loader(model_fold, val_loader)

    print(f"  Validation CE={val_loss:.4f} | Validation Acc={val_acc*100:.4f}%\n")
    fold_results.append(val_acc)

# -----------------------------
# Summary
# -----------------------------
fold_results = np.array(fold_results)
mean_acc = fold_results.mean() * 100
std_acc = fold_results.std() * 100

print("Fold Accuracies (%):", [round(x*100, 4) for x in fold_results])
print(f"Mean Accuracy = {mean_acc:.4f}%")
print(f"Std Accuracy  = {std_acc:.4f}%")

# -----------------------------
# Save text results
# -----------------------------
txt_path = os.path.join(FIG_DIR, "kfold_results.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    for i, acc in enumerate(fold_results, start=1):
        f.write(f"Fold {i}: {acc*100:.6f}%\n")
    f.write(f"\nMean Accuracy: {mean_acc:.6f}%\n")
    f.write(f"Std Accuracy: {std_acc:.6f}%\n")

# -----------------------------
# Plot fold accuracies
# -----------------------------
fig_path = os.path.join(FIG_DIR, "kfold_accuracy_1200dpi.png")

plt.figure(figsize=(6, 6))
x = np.arange(1, N_SPLITS + 1)
y = fold_results * 100

plt.plot(x, y, marker="o", linewidth=3)

for i, v in enumerate(y):
    plt.text(x[i], v + 0.05, f"{v:.2f}", ha="center", fontsize=12, fontweight="bold")

plt.xlabel("Fold", fontsize=16, fontweight="bold")
plt.ylabel("Accuracy (%)", fontsize=16, fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_path, dpi=1200, bbox_inches="tight")
plt.close()

print("\nSaved figure:", fig_path)
print("Saved text  :", txt_path)

print("\nSTEP 12 complete.")

Device: cpu
Total source samples: 22099
Total source labels : 22099

Running 5-Fold Cross Validation...

Fold 1/5
  Validation CE=0.0224 | Validation Acc=99.9774%

Fold 2/5
  Validation CE=0.0178 | Validation Acc=100.0000%

Fold 3/5
  Validation CE=0.0367 | Validation Acc=99.6380%

Fold 4/5
  Validation CE=0.0181 | Validation Acc=100.0000%

Fold 5/5
  Validation CE=0.0178 | Validation Acc=100.0000%

Fold Accuracies (%): [99.9774, 100.0, 99.638, 100.0, 100.0]
Mean Accuracy = 99.9231%
Std Accuracy  = 0.1428%

Saved figure: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\kfold_accuracy_1200dpi.png
Saved text  : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\kfold_results.txt

STEP 12 complete.


In [28]:
# =========================================================
# STEP 13: Ablation Study
# =========================================================

import os
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ABL_DIR = os.path.join(FIG_DIR, "ablation_study")
os.makedirs(ABL_DIR, exist_ok=True)

# -----------------------------
# Load source-pretrained checkpoint
# -----------------------------
src_ckpt_path = os.path.join(CKPT_DIR, "best_source_pretrain.pt")
src_ckpt = torch.load(src_ckpt_path, map_location=device)

# -----------------------------
# Evaluation function
# -----------------------------
@torch.no_grad()
def evaluate_target_acc(model_eval, loader):
    model_eval.eval()
    correct = 0
    total = 0

    for xb, xb_aug, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model_eval(xb)
        pred = torch.argmax(logits, dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    return correct / max(total, 1)

# -----------------------------
# Build pseudo datasets
# -----------------------------
pseudo_labels = np.load(os.path.join(NPY_DIR, "pseudo_labels.npy")).astype(np.int64)
pseudo_conf   = np.load(os.path.join(NPY_DIR, "pseudo_confidence.npy")).astype(np.float32)

tgt_samples = tgt_train_dataset.samples.astype(np.float32)

assert len(tgt_samples) == len(pseudo_labels) == len(pseudo_conf)

class PlainPseudoDataset(Dataset):
    def __init__(self, samples, labels):
        self.samples = samples
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx]
        y = self.labels[idx]

        x = torch.tensor(x).float().unsqueeze(0)
        y = torch.tensor(y).long()
        return x, y

class WeightedPseudoDataset(Dataset):
    def __init__(self, samples, labels, weights):
        self.samples = samples
        self.labels = labels
        self.weights = weights

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx]
        y = self.labels[idx]
        w = self.weights[idx]

        x = torch.tensor(x).float().unsqueeze(0)
        y = torch.tensor(y).long()
        w = torch.tensor(w).float()
        return x, y, w

def augment_signal_local(x):
    x_aug = x.copy()
    noise = np.random.normal(0, 0.01, size=x_aug.shape)
    x_aug += noise
    scale = np.random.uniform(0.9, 1.1)
    x_aug *= scale
    return x_aug.astype(np.float32)

class WeightedConsistencyDataset(Dataset):
    def __init__(self, samples, labels, weights):
        self.samples = samples
        self.labels = labels
        self.weights = weights

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx]
        x_aug = augment_signal_local(x)
        y = self.labels[idx]
        w = self.weights[idx]

        x = torch.tensor(x).float().unsqueeze(0)
        x_aug = torch.tensor(x_aug).float().unsqueeze(0)
        y = torch.tensor(y).long()
        w = torch.tensor(w).float()

        return x, x_aug, y, w

plain_ds = PlainPseudoDataset(tgt_samples, pseudo_labels)
weighted_ds = WeightedPseudoDataset(tgt_samples, pseudo_labels, pseudo_conf)
weighted_consistency_ds = WeightedConsistencyDataset(tgt_samples, pseudo_labels, pseudo_conf)

plain_loader = DataLoader(plain_ds, batch_size=128, shuffle=True, drop_last=True)
weighted_loader = DataLoader(weighted_ds, batch_size=128, shuffle=True, drop_last=True)
cons_loader = DataLoader(weighted_consistency_ds, batch_size=128, shuffle=True, drop_last=True)

# -----------------------------
# Ablation 1: Source Only
# -----------------------------
model_source = FaultNet(feat_dim=128, n_classes=4).to(device)
model_source.load_state_dict(src_ckpt["model_state"])
acc_source = evaluate_target_acc(model_source, tgt_te_loader)
print(f"Source Only Acc = {acc_source*100:.4f}%")

# -----------------------------
# Ablation 2: Plain pseudo-label fine-tuning
# -----------------------------
model_plain = FaultNet(feat_dim=128, n_classes=4).to(device)
model_plain.load_state_dict(src_ckpt["model_state"])
opt_plain = torch.optim.Adam(model_plain.parameters(), lr=5e-4, weight_decay=1e-4)

EPOCHS_ABL = 4

for epoch in range(EPOCHS_ABL):
    model_plain.train()

    for x, y in plain_loader:
        x = x.to(device)
        y = y.to(device)

        feat, logits = model_plain(x)
        loss = F.cross_entropy(logits, y)

        opt_plain.zero_grad()
        loss.backward()
        opt_plain.step()

acc_plain = evaluate_target_acc(model_plain, tgt_te_loader)
print(f"Plain Pseudo Fine-Tuning Acc = {acc_plain*100:.4f}%")

# -----------------------------
# Ablation 3: Confidence-weighted adaptation
# -----------------------------
model_weighted = FaultNet(feat_dim=128, n_classes=4).to(device)
model_weighted.load_state_dict(src_ckpt["model_state"])
opt_weighted = torch.optim.Adam(model_weighted.parameters(), lr=5e-4, weight_decay=1e-4)

for epoch in range(EPOCHS_ABL):
    model_weighted.train()

    for x, y, w in weighted_loader:
        x = x.to(device)
        y = y.to(device)
        w = w.to(device)

        feat, logits = model_weighted(x)
        ce_each = F.cross_entropy(logits, y, reduction="none")
        loss = (w * ce_each).mean()

        opt_weighted.zero_grad()
        loss.backward()
        opt_weighted.step()

acc_weighted = evaluate_target_acc(model_weighted, tgt_te_loader)
print(f"Confidence-Weighted Adaptation Acc = {acc_weighted*100:.4f}%")

# -----------------------------
# Ablation 4: Weighted + consistency
# -----------------------------
model_cons = FaultNet(feat_dim=128, n_classes=4).to(device)
model_cons.load_state_dict(src_ckpt["model_state"])
opt_cons = torch.optim.Adam(model_cons.parameters(), lr=5e-4, weight_decay=1e-4)

lambda_cons = 0.5

for epoch in range(EPOCHS_ABL):
    model_cons.train()

    for x, x_aug, y, w in cons_loader:
        x = x.to(device)
        x_aug = x_aug.to(device)
        y = y.to(device)
        w = w.to(device)

        feat1, logits1 = model_cons(x)
        feat2, logits2 = model_cons(x_aug)

        ce_each = F.cross_entropy(logits1, y, reduction="none")
        loss_ce = (w * ce_each).mean()

        p1 = F.softmax(logits1, dim=1)
        p2 = F.softmax(logits2, dim=1)
        loss_cons = F.kl_div(torch.log(p1 + 1e-12), p2.detach(), reduction="batchmean")

        loss = loss_ce + lambda_cons * loss_cons

        opt_cons.zero_grad()
        loss.backward()
        opt_cons.step()

acc_cons = evaluate_target_acc(model_cons, tgt_te_loader)
print(f"Weighted + Consistency Acc = {acc_cons*100:.4f}%")

# -----------------------------
# Save results
# -----------------------------
ablation_names = [
    "Source Only",
    "Plain Pseudo FT",
    "Conf-Weighted",
    "Conf-Weighted + Consistency"
]
ablation_accs = [
    acc_source * 100,
    acc_plain * 100,
    acc_weighted * 100,
    acc_cons * 100
]

txt_path = os.path.join(ABL_DIR, "ablation_results.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    for name, acc in zip(ablation_names, ablation_accs):
        f.write(f"{name}: {acc:.6f}%\n")

fig_path = os.path.join(ABL_DIR, "ablation_barplot_1200dpi.png")

plt.figure(figsize=(9, 6))
bars = plt.bar(ablation_names, ablation_accs)

for bar, val in zip(bars, ablation_accs):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.05,
        f"{val:.2f}",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

plt.ylabel("Accuracy (%)", fontsize=16, fontweight="bold")
plt.xticks(rotation=15, fontsize=12, fontweight="bold")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(fig_path, dpi=1200, bbox_inches="tight")
plt.close()

print("\nSaved ablation text to:")
print(txt_path)
print("Saved ablation figure to:")
print(fig_path)

print("\nSTEP 13 complete.")

Device: cpu


C:\Users\saifu\AppData\Local\Temp\ipykernel_24196\3252303737.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  src_ckpt = torch.load(src_ckpt_path, map_location=device)


Source Only Acc = 99.6753%
Plain Pseudo Fine-Tuning Acc = 91.6280%
Confidence-Weighted Adaptation Acc = 99.7449%
Weighted + Consistency Acc = 98.7709%

Saved ablation text to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\ablation_study\ablation_results.txt
Saved ablation figure to:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\ablation_study\ablation_barplot_1200dpi.png

STEP 13 complete.


In [29]:
# =========================================================
# STEP 14: Before vs After Adaptation comparison figures
# =========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

COMPARE_DIR = os.path.join(FIG_DIR, "before_after_comparison")
os.makedirs(COMPARE_DIR, exist_ok=True)

class_names = ["BF", "GF", "N", "TF"]
n_classes = len(class_names)

# -----------------------------
# Helper: collect outputs
# -----------------------------
@torch.no_grad()
def collect_outputs(model_eval, loader):
    model_eval.eval()

    all_ytrue = []
    all_ypred = []
    all_yprob = []
    all_feat = []

    for xb, xb_aug, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        feat, logits = model_eval(xb)
        prob = F.softmax(logits, dim=1)
        pred = torch.argmax(prob, dim=1)

        all_ytrue.append(yb.cpu().numpy())
        all_ypred.append(pred.cpu().numpy())
        all_yprob.append(prob.cpu().numpy())
        all_feat.append(feat.cpu().numpy())

    y_true = np.concatenate(all_ytrue, axis=0).astype(np.int64)
    y_pred = np.concatenate(all_ypred, axis=0).astype(np.int64)
    y_prob = np.concatenate(all_yprob, axis=0).astype(np.float32)
    feat = np.concatenate(all_feat, axis=0).astype(np.float32)

    acc = (y_true == y_pred).mean()

    return y_true, y_pred, y_prob, feat, acc

# -----------------------------
# Load BEFORE model (source only)
# -----------------------------
src_ckpt_path = os.path.join(CKPT_DIR, "best_source_pretrain.pt")
src_ckpt = torch.load(src_ckpt_path, map_location=device)

model_before = FaultNet(feat_dim=128, n_classes=4).to(device)
model_before.load_state_dict(src_ckpt["model_state"])

# -----------------------------
# Load AFTER model (adapted)
# -----------------------------
aft_ckpt_path = os.path.join(CKPT_DIR, "weighted_stage2_best.pt")
aft_ckpt = torch.load(aft_ckpt_path, map_location=device)

model_after = FaultNet(feat_dim=128, n_classes=4).to(device)
model_after.load_state_dict(aft_ckpt["model_state"])

# -----------------------------
# Collect outputs
# -----------------------------
print("\nCollecting BEFORE outputs...")
y_true_b, y_pred_b, y_prob_b, feat_b, acc_b = collect_outputs(model_before, tgt_te_loader)

print("Collecting AFTER outputs...")
y_true_a, y_pred_a, y_prob_a, feat_a, acc_a = collect_outputs(model_after, tgt_te_loader)

print(f"\nBefore accuracy: {acc_b*100:.4f}%")
print(f"After accuracy : {acc_a*100:.4f}%")

# -----------------------------
# Save comparison summary
# -----------------------------
summary_txt = os.path.join(COMPARE_DIR, "before_after_summary.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"Before Adaptation Accuracy: {acc_b*100:.6f}%\n")
    f.write(f"After Adaptation Accuracy : {acc_a*100:.6f}%\n")

# -----------------------------
# Helper: confusion matrix
# -----------------------------
def save_confusion(cm, name):
    path = os.path.join(COMPARE_DIR, f"{name}_confusion_matrix_1200dpi.png")

    plt.figure(figsize=(7, 6))
    plt.imshow(cm, cmap="YlGnBu")

    ticks = np.arange(n_classes)
    plt.xticks(ticks, class_names, fontsize=15, fontweight="bold")
    plt.yticks(ticks, class_names, fontsize=15, fontweight="bold")

    plt.gca().set_xticks(np.arange(-.5, n_classes, 1), minor=True)
    plt.gca().set_yticks(np.arange(-.5, n_classes, 1), minor=True)
    plt.grid(which="minor", color="black", linestyle="-", linewidth=1.5)
    plt.tick_params(which="minor", bottom=False, left=False)

    thresh = cm.max() * 0.5 if cm.max() > 0 else 0.5

    for i in range(n_classes):
        for j in range(n_classes):
            plt.text(
                j, i, f"{cm[i, j]}",
                ha="center", va="center",
                fontsize=17, fontweight="bold",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.xlabel("Predicted Class", fontsize=17, fontweight="bold")
    plt.ylabel("True Class", fontsize=17, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=1200, bbox_inches="tight")
    plt.close()

    return path

# -----------------------------
# Save confusion matrices
# -----------------------------
cm_before = confusion_matrix(y_true_b, y_pred_b)
cm_after  = confusion_matrix(y_true_a, y_pred_a)

cm_before_path = save_confusion(cm_before, "before")
cm_after_path  = save_confusion(cm_after, "after")

print("\nSaved confusion matrices:")
print(cm_before_path)
print(cm_after_path)

# -----------------------------
# Helper: ROC
# -----------------------------
def save_roc(y_true, y_prob, name):
    path = os.path.join(COMPARE_DIR, f"{name}_roc_1200dpi.png")

    y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
    line_styles = ["-", "--", ":", "-."]

    plt.figure(figsize=(8, 6.5))

    auc_values = []

    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, c], y_prob[:, c])
        roc_auc = auc(fpr, tpr)
        auc_values.append(roc_auc)

        plt.plot(
            fpr, tpr,
            linestyle=line_styles[c],
            linewidth=3,
            label=f"{class_names[c]} (AUC={roc_auc:.4f})"
        )

    plt.plot([0, 1], [0, 1], "k--", linewidth=2, label="Chance")
    plt.xlabel("False Positive Rate", fontsize=16, fontweight="bold")
    plt.ylabel("True Positive Rate", fontsize=16, fontweight="bold")

    leg = plt.legend(fontsize=12, frameon=True)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1.5)

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=1200, bbox_inches="tight")
    plt.close()

    txt_path = os.path.join(COMPARE_DIR, f"{name}_auc_scores.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for i, cname in enumerate(class_names):
            f.write(f"{cname}: {auc_values[i]:.6f}\n")

    return path, txt_path

roc_before_path, auc_before_path = save_roc(y_true_b, y_prob_b, "before")
roc_after_path, auc_after_path = save_roc(y_true_a, y_prob_a, "after")

print("\nSaved ROC figures:")
print(roc_before_path)
print(roc_after_path)

# -----------------------------
# Helper: t-SNE
# -----------------------------
def save_tsne(features, y_true, name):
    path = os.path.join(COMPARE_DIR, f"{name}_tsne_1200dpi.png")

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        init="pca",
        learning_rate="auto",
        random_state=42
    )
    Z = tsne.fit_transform(features)

    markers = ["o", "s", "^", "D"]

    plt.figure(figsize=(8, 7))
    for c, cname in enumerate(class_names):
        idx = (y_true == c)
        plt.scatter(
            Z[idx, 0], Z[idx, 1],
            s=70,
            marker=markers[c],
            edgecolors="black",
            linewidths=0.8,
            alpha=0.9,
            label=cname
        )

    plt.xlabel("t-SNE Dimension 1", fontsize=16, fontweight="bold")
    plt.ylabel("t-SNE Dimension 2", fontsize=16, fontweight="bold")

    leg = plt.legend(fontsize=12, frameon=True)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(path, dpi=1200, bbox_inches="tight")
    plt.close()

    np.save(os.path.join(COMPARE_DIR, f"{name}_tsne_embedding.npy"), Z)

    return path

tsne_before_path = save_tsne(feat_b, y_true_b, "before")
tsne_after_path  = save_tsne(feat_a, y_true_a, "after")

print("\nSaved t-SNE figures:")
print(tsne_before_path)
print(tsne_after_path)

print("\nSaved summary text:")
print(summary_txt)

print("\nSTEP 14 complete.")

Device: cpu



C:\Users\saifu\AppData\Local\Temp\ipykernel_24196\219080173.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  src_ckpt = torch.load(src_ckpt_path, map_location=device)
C:


Before accuracy: 99.6753%
After accuracy : 99.9768%

Saved confusion matrices:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\before_confusion_matrix_1200dpi.png
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\after_confusion_matrix_1200dpi.png

Saved ROC figures:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\before_roc_1200dpi.png
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\after_roc_1200dpi.png

Saved t-SNE figures:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\before_tsne_1200dpi.png
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\after_tsne_1200dpi.png

Saved summary text:
G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\before_after_comparison\before_after_summary.txt

STEP 14 complete.


In [30]:
# =========================================================
# STEP 15: Manuscript-ready summary table
# =========================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SUMMARY_DIR = os.path.join(FIG_DIR, "summary_tables")
os.makedirs(SUMMARY_DIR, exist_ok=True)

print("SUMMARY_DIR:", SUMMARY_DIR)

# -----------------------------
# Helper functions
# -----------------------------
def read_text_if_exists(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    return None

def extract_first_float_after_keyword(text, keyword):
    if text is None:
        return None
    pattern = rf"{re.escape(keyword)}\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)"
    m = re.search(pattern, text)
    if m:
        return float(m.group(1))
    return None

def safe_format(x, nd=4, suffix=""):
    if x is None:
        return "N/A"
    return f"{x:.{nd}f}{suffix}"

# -----------------------------
# 1. Source-only vs adapted
# -----------------------------
before_after_txt = os.path.join(FIG_DIR, "before_after_comparison", "before_after_summary.txt")
before_after_text = read_text_if_exists(before_after_txt)

src_only_acc = None
adapted_acc = None

if before_after_text is not None:
    src_only_acc = extract_first_float_after_keyword(before_after_text, "Before Adaptation Accuracy")
    adapted_acc = extract_first_float_after_keyword(before_after_text, "After Adaptation Accuracy")

# fallback using known checkpoint summary if needed
if adapted_acc is None:
    weighted_ckpt_path = os.path.join(CKPT_DIR, "weighted_stage2_best.pt")
    if os.path.exists(weighted_ckpt_path):
        adapted_acc = None  # keep N/A if not directly available

# -----------------------------
# 2. SNR results
# -----------------------------
snr_txt_candidates = [
    os.path.join(FIG_DIR, "snr_experiments", "snr_results.txt"),
    os.path.join(FIG_DIR, "snr_experiments_20_to_40db", "snr_results_20_to_40db.txt")
]

snr_results = {}
for p in snr_txt_candidates:
    txt = read_text_if_exists(p)
    if txt is not None:
        for line in txt.splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                v = v.strip().replace("%", "")
                try:
                    snr_results[k.strip()] = float(v)
                except:
                    pass

# -----------------------------
# 3. Small labeled source data
# -----------------------------
small_src_txt = os.path.join(FIG_DIR, "small_source_data_results.txt")
small_src_text = read_text_if_exists(small_src_txt)

small_src_results = {}
if small_src_text is not None:
    for line in small_src_text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            v = v.strip().replace("%", "")
            try:
                small_src_results[k.strip()] = float(v)
            except:
                pass

# -----------------------------
# 4. K-fold results
# -----------------------------
kfold_txt = os.path.join(FIG_DIR, "kfold_results.txt")
kfold_text = read_text_if_exists(kfold_txt)

kfold_mean = None
kfold_std = None

if kfold_text is not None:
    kfold_mean = extract_first_float_after_keyword(kfold_text, "Mean Accuracy")
    kfold_std = extract_first_float_after_keyword(kfold_text, "Std Accuracy")

# -----------------------------
# 5. Ablation results
# -----------------------------
ablation_txt = os.path.join(FIG_DIR, "ablation_study", "ablation_results.txt")
ablation_text = read_text_if_exists(ablation_txt)

ablation_results = {}
if ablation_text is not None:
    for line in ablation_text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            v = v.strip().replace("%", "")
            try:
                ablation_results[k.strip()] = float(v)
            except:
                pass

# -----------------------------
# 6. t-SNE / feature metrics
# -----------------------------
metric_txt = os.path.join(FIG_DIR, "final_eval", "tsne_and_feature_metrics.txt")
metric_text = read_text_if_exists(metric_txt)

metrics = {}
if metric_text is not None:
    for line in metric_text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            try:
                metrics[k.strip()] = float(v.strip())
            except:
                pass

# -----------------------------
# Build summary dataframe
# -----------------------------
rows = []

rows.append({
    "Category": "Main Performance",
    "Item": "Source Only Accuracy",
    "Value": safe_format(src_only_acc, 4, "%")
})
rows.append({
    "Category": "Main Performance",
    "Item": "Adapted Accuracy",
    "Value": safe_format(adapted_acc, 4, "%")
})

for k in ["Clean", "40dB", "40 dB", "30dB", "30 dB", "20dB", "20 dB", "10dB", "10 dB", "5dB", "5 dB", "0dB", "0 dB"]:
    if k in snr_results:
        rows.append({
            "Category": "SNR Robustness",
            "Item": f"Accuracy at {k}",
            "Value": safe_format(snr_results[k], 4, "%")
        })

for k in ["20%", "40%", "60%", "80%", "100%"]:
    rows.append({
        "Category": "Limited Source Labels",
        "Item": f"Target Accuracy with {k} Source Labels",
        "Value": safe_format(small_src_results.get(k, None), 4, "%")
    })

rows.append({
    "Category": "K-Fold Validation",
    "Item": "Mean Accuracy",
    "Value": safe_format(kfold_mean, 4, "%")
})
rows.append({
    "Category": "K-Fold Validation",
    "Item": "Std Accuracy",
    "Value": safe_format(kfold_std, 4, "%")
})

for k in [
    "Source Only",
    "Plain Pseudo FT",
    "Conf-Weighted",
    "Conf-Weighted + Consistency"
]:
    rows.append({
        "Category": "Ablation Study",
        "Item": k,
        "Value": safe_format(ablation_results.get(k, None), 4, "%")
    })

for k in [
    "Silhouette Score",
    "Davies-Bouldin Index",
    "Calinski-Harabasz Score"
]:
    if k in metrics:
        rows.append({
            "Category": "Feature / t-SNE Metrics",
            "Item": k,
            "Value": safe_format(metrics[k], 6, "")
        })

df = pd.DataFrame(rows)

# -----------------------------
# Save CSV
# -----------------------------
csv_path = os.path.join(SUMMARY_DIR, "manuscript_summary_table.csv")
df.to_csv(csv_path, index=False)

# -----------------------------
# Save TXT
# -----------------------------
txt_path = os.path.join(SUMMARY_DIR, "manuscript_summary_table.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(df.to_string(index=False))

# -----------------------------
# Save figure-table
# -----------------------------
fig_path = os.path.join(SUMMARY_DIR, "manuscript_summary_table_1200dpi.png")

n_rows = len(df) + 1
fig_h = max(8, 0.45 * n_rows)

plt.figure(figsize=(14, fig_h))
plt.axis("off")

table = plt.table(
    cellText=df.values,
    colLabels=df.columns,
    loc="center",
    cellLoc="left",
    colLoc="left"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.8)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#D9EAF7")

plt.tight_layout()
plt.savefig(fig_path, dpi=1200, bbox_inches="tight")
plt.close()

print("\nSaved files:")
print("CSV :", csv_path)
print("TXT :", txt_path)
print("FIG :", fig_path)

print("\nPreview:")
print(df.to_string(index=False))

print("\nSTEP 15 complete.")

SUMMARY_DIR: G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\summary_tables

Saved files:
CSV : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\summary_tables\manuscript_summary_table.csv
TXT : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\summary_tables\manuscript_summary_table.txt
FIG : G:\New Implemented Papers\Cutting Tool Paper\1320_1440\figs\summary_tables\manuscript_summary_table_1200dpi.png

Preview:
               Category                                    Item        Value
       Main Performance                    Source Only Accuracy     99.6753%
       Main Performance                        Adapted Accuracy     99.9768%
         SNR Robustness                       Accuracy at Clean     99.9768%
         SNR Robustness                       Accuracy at 40 dB     99.9768%
         SNR Robustness                       Accuracy at 30 dB     99.7449%
         SNR Robustness                        Accuracy at 20dB     84.0445%
         SN